# ⚖️ GIADA roadmap Task 15 — decomposizione contro corrente diretta
Tre bracci, stesso tensore causale di 10 valori, dati e minibatch appaiati: (A) percorso di V e gate m/h **appresi** → formula analitica della corrente; (B) corrente appresa direttamente; (C) corrente diretta con percorso V e gate m/h come target ausiliari. L'integrazione analitica dei gate è solo un probe, non il braccio A. Il target è la corrente **analitica all'endpoint** del millisecondo, non `ica` nativa sfasata (Task13). Tutto si svolge nel playground Ca-HVA+pas a un compartimento. Nessuna promozione a Gate C o neurone completo.


In [ ]:
from pathlib import Path
import base64,json,os,shutil,subprocess,sys
from IPython.display import Javascript,display
WORK=Path('/kaggle/working/giada_roadmap_task15');REPO=WORK/'giada'
assert not WORK.exists(),f'Workspace già presente: {WORK}. Avvia una sessione pulita.'
WORK.mkdir(parents=True)
subprocess.run(['git','clone','https://github.com/Zagred47/giada.git',str(REPO)],check=True)
subprocess.run(['git','-C',str(REPO),'fetch','origin','codex/surrogate-validity-audit'],check=True)
subprocess.run(['git','-C',str(REPO),'checkout','--detach','FETCH_HEAD'],check=True)
REVISION=subprocess.check_output(['git','-C',str(REPO),'rev-parse','HEAD'],text=True).strip()
assert (REPO/'src/giada_teacher/roadmap_current_architecture_comparison.py').is_file(),'Revisione GIADA senza Task15.'
sys.path.insert(0,str(REPO))
for name in [n for n in list(sys.modules) if n=='src' or n.startswith('src.')]:del sys.modules[name]
from src.giada_teacher.roadmap_current_architecture_comparison import CurrentArchitectureConfig,verified_task14_report,train_and_freeze,evaluate_frozen
print({'revision':REVISION})


## 📁 Unico artefatto esterno
Aggiungi il Dataset Kaggle contenente `giada_roadmap_task14_parameter_variation.zip` oppure la sua cartella estratta. L'hash del report viene verificato. Se montato sotto un nome inatteso, imposta `GIADA_TASK14_ARTIFACT` al percorso ZIP/cartella. Non servono dataset HayFlow multi-GB né checkpoint Task11: questi tre bracci si addestrano ex novo con input allineati.


In [ ]:
INPUT=Path('/kaggle/input');override=os.environ.get('GIADA_TASK14_ARTIFACT')
candidates=[Path(override).expanduser()] if override else []
if INPUT.is_dir():
 candidates += [p for p in INPUT.rglob('*.zip') if 'task14' in str(p).lower() or 'parameter-variation' in str(p).lower() or 'parameter_variation' in str(p).lower()]
 candidates += [p.parent for p in INPUT.rglob('task14_parameter_matrix_report.json')]
TASK14=None;contract14=None
for candidate in dict.fromkeys(candidates):
 if not candidate.exists():continue
 try:contract14=verified_task14_report(candidate);TASK14=candidate.resolve();break
 except (ValueError,KeyError,FileNotFoundError,OSError):continue
assert TASK14 is not None,'Task14 esatta non trovata: aggiungi giada_roadmap_task14_parameter_variation.zip agli Input o imposta GIADA_TASK14_ARTIFACT.'
display({'source':str(TASK14),'verified_task14':contract14})


## 🧠 Train + sviluppo, poi freeze
192 episodi train seed15011 e 48 development seed15029; ECa 100–140 mV, gbar×0.5/1/1.5, gate iniziali perturbati e una frazione di canali off visibili. Tre seed GPU (17/29/43), 400 step e checkpoint valutativi 50/100/200/400. Stessi minibatch per i tre bracci entro ciascun seed. La selezione prende solo il seed con minore RMSE di corrente sul development **allo step 400**; gli step intermedi sono una mini scaling law, non nuove occasioni di scegliere il modello. Sealed e perturbazioni sono ancora chiusi.


In [ ]:
os.environ.setdefault('CUBLAS_WORKSPACE_CONFIG',':4096:8')
import torch
assert torch.cuda.is_available(),'La Task15 preregistrata richiede una GPU CUDA Kaggle.'
DEVICE='cuda';config=CurrentArchitectureConfig();config.validate()
OUTPUT=Path('/kaggle/working/artifacts/giada_roadmap_task15_current_architecture_comparison')
assert not OUTPUT.exists(),f'Output già presente: {OUTPUT}. Avvia una sessione nuova per evitare mix di checkpoint.'
print({'gpu':torch.cuda.get_device_name(0),'seeds':config.seeds,'steps':config.steps},flush=True)
freeze=train_and_freeze(TASK14,OUTPUT,config,device=DEVICE)
display({'selected':freeze['selected'],'freeze_sha256':freeze['freeze_sha256'],'sealed_accessed':freeze['sealed_test_accessed'],'paired_minibatches':freeze['paired_minibatch_indices_within_seed']})
assert not freeze['sealed_test_accessed'] and not freeze['counterfactual_test_accessed']


## 🔒 Valutazione sealed e perturbazioni una sola volta
48 episodi sealed seed15059 e 48 episodi controfattuali seed15159, generati **solo dopo** il freeze. Metriche: RMSE corrente globale/attiva/off, voltaggio/gate ove disponibili, vincolo di corrente zero da maschera, delta di corrente e V rispetto alla baseline, rollout 16ms soltanto dove esiste una transizione di stato. Il braccio diretto non ha rollout; il percorso ausiliario non è guidato dal suo head di corrente.


In [ ]:
final=evaluate_frozen(freeze,OUTPUT,config,device=DEVICE,code_revision=REVISION)
compact={arm:{'current_rmse':row['all_rmse_ma_cm2'],'active_rmse':row['active_rmse_ma_cm2'],'off_rmse':row['off_rmse_ma_cm2'],'voltage_rmse':row['voltage_rmse_mv'],'rollout':row['recursive_16ms']} for arm,row in final['sealed_one_step_and_available_rollout'].items()}
display({'valid':final['valid'],'same_input':final['common_numeric_input_tensor'],'selected':final['selected'],'sealed':compact,'paired_conditions':list(final['paired_counterfactuals'])})
if not final['valid']:print('ATTENZIONE: report non valido; scarica comunque lo ZIP diagnostico, non interpretare il ranking.')


## 📦 Scarica il risultato
ZIP piccolo con freeze, checkpoint e report. Metodo Blob/base64 concordato: non usa FileLink né il pannello Files.


In [ ]:
archive=Path(shutil.make_archive('/kaggle/working/giada_roadmap_task15_current_architecture_comparison','zip',OUTPUT.parent,OUTPUT.name))
payload=base64.b64encode(archive.read_bytes()).decode('ascii')
display(Javascript(f"""const b=atob('{payload}');const a=new Uint8Array(b.length);for(let i=0;i<b.length;i++)a[i]=b.charCodeAt(i);const u=URL.createObjectURL(new Blob([a],{{type:'application/zip'}}));const l=document.createElement('a');l.href=u;l.download='{archive.name}';document.body.appendChild(l);l.click();l.remove();setTimeout(()=>URL.revokeObjectURL(u),1000);"""))
print({'archive':archive.name,'size_mib':round(archive.stat().st_size/2**20,2)})
